In [5]:
import json
import joblib
import numpy as np
import pandas as pd

In [ ]:
ARTIFACTS_DIR = r"Path\prototype\artifacts"

HEART_MODEL_PATH  = ARTIFACTS_DIR+r"\artifacts_heart\random_forest_pipeline.pkl"
SYMPTOMS_MODEL_PATH  = ARTIFACTS_DIR+r"\artifacts_disease_prediction\logreg_multiclass_pipeline.pkl"

CONFIG_HEART_PATH = ARTIFACTS_DIR +r"\artifacts_heart\config.json"
CONFIG_DISEASE_PREDICTION_PATH = ARTIFACTS_DIR +r"\artifacts_disease_prediction\config.json"


In [12]:
heart_model = joblib.load(HEART_MODEL_PATH)
symptom_model = joblib.load(SYMPTOMS_MODEL_PATH)

In [16]:
with open(CONFIG_DISEASE_PREDICTION_PATH, "r") as f:
    symptom_config = json.load(f)

In [17]:
label_mappping = symptom_config["label_mapping"]
inv_label_mapping = {v:k for k,v in label_mappping.items()}
symptom_features = symptom_config["features"]

In [18]:
def predict_heart_risk(payload: dict):
    X = pd.DataFrame([payload])
    proba = heart_model.predict_proba(X)[:,1][0]
    pred = int(proba >= 0.5)

    return {
        "prediction": pred,
        "probability_disease": float(proba)
    }


In [19]:
def predict_symptom_disease(payload:dict):
    row = {feat:int(payload.get(feat,0)) for feat in symptom_features}
    X =  pd.DataFrame([row])

    probs = symptom_model.predict_proba(X)[0]
    top_idx = int(np.argmax(probs))
    prob_list = sorted(
        [(inv_label_mapping[i], float(p)) for i,p in enumerate(probs)], 
        key = lambda x:x[1], 
        reverse = True
    )

    return {
        "prediction":inv_label_mapping[top_idx], 
        "probabilities":prob_list
    }


In [20]:
# Heart example
print(predict_heart_risk({
    "age": 55, "trestbps": 140, "chol": 220, "thalch": 150, "oldpeak": 1.2, "ca": 0,
    "sex": "Male", "dataset": "Cleveland", "cp": "typical angina", "fbs": False,
    "restecg": "normal", "exang": False, "slope": "flat", "thal": "normal"
}))

# Symptoms example
print(predict_symptom_disease({
    "fever": 1, "cough": 1, "headache": 1, "fatigue": 1
}))


{'prediction': 0, 'probability_disease': 0.3775}
{'prediction': 'Malaria', 'probabilities': [('Malaria', 0.5330421846581198), ('Pneumonia', 0.4290667382505904), ('Typhoid', 0.03789107709128985)]}
